# MinIO smoke test
Quick check that Spark can write and read files through the S3A connector.

What happens here:
- configure Spark to talk to the local MinIO endpoint
- write a small DataFrame as Parquet
- read it back to confirm connectivity


## 1) Configure the S3A connection
Update the bucket or path if you want to isolate your own sandbox inside MinIO.


In [1]:
import os
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("minio-quick-check").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
bucket = os.getenv("MINIO_BUCKET", "mydatalab")
target_path = f"s3a://{bucket}/spark-test"

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://127.0.0.1:9000")
hconf.set("fs.s3a.access.key", access_key)
hconf.set("fs.s3a.secret.key", secret_key)
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hconf.set("fs.s3a.connection.ssl.enabled", "false")

print("MinIO endpoint: http://127.0.0.1:9000")
print(f"Target path: {target_path}")


MinIO endpoint: http://127.0.0.1:9000
Target path: s3a://mydatalab/spark-test


## 2) Write a tiny dataset to MinIO
A simple DataFrame with a timestamp column so every run is easy to spot.


In [2]:
sample = [("pavel", 1), ("mark", 2), ("dana", 3)]
df = spark.createDataFrame(sample, ["name", "value"]).withColumn("ts", F.current_timestamp())

print("Data going to MinIO:")
df.orderBy("value").show(truncate=False)

df.write.mode("overwrite").parquet(target_path)


Data going to MinIO:
+-----+-----+--------------------------+
|name |value|ts                        |
+-----+-----+--------------------------+
|pavel|1    |2025-11-25 01:28:08.830404|
|mark |2    |2025-11-25 01:28:08.830404|
|dana |3    |2025-11-25 01:28:08.830404|
+-----+-----+--------------------------+



## 3) Read it back and validate
If you see the same rows below, the S3A connector is working end-to-end.


In [3]:
df_read = spark.read.parquet(target_path)

print("Data read back from MinIO:")
df_read.orderBy("value").show(truncate=False)

print("Row count:")
df_read.groupBy().count().show()


Data read back from MinIO:
+-----+-----+--------------------------+
|name |value|ts                        |
+-----+-----+--------------------------+
|pavel|1    |2025-11-25 01:28:22.633072|
|mark |2    |2025-11-25 01:28:22.633072|
|dana |3    |2025-11-25 01:28:22.633072|
+-----+-----+--------------------------+

Row count:
+-----+
|count|
+-----+
|    3|
+-----+



## Next steps
- change `target_path` to a personal prefix so runs do not collide with others
- swap `parquet` for other formats to validate codecs and permissions
- point the endpoint to a different MinIO/S3 host to reuse this smoke test
